# H₄ lineal con VQE: curva de energía, $R_e$ y $D_e$

In [ ]:
# Instalación de dependencias (versiones fijadas con las que desarrollamos)
%pip install -q qiskit==2.6.2 qiskit-aer==0.18.4 qiskit-nature==0.8.0 pyscf==2.15.1

In [ ]:
import time
_notebook_t0 = time.time()

In [ ]:
# ---- src/hamiltonian.py ----
"""Química de H4 lineal (STO-3G): de la distancia d al hamiltoniano en qubits.

Dueño: Gonza. Interfaz acordada en docs/task_split.md:

    build_problem(d) -> dict con hamiltoniano, bitstring de HF, energías de
    referencia (HF, CCSD, FCI), amplitudes t1/t2 de CCSD y los operadores N y S².

Convenciones (las mismas que usa el workshop de Química Cuántica y qiskit-nature
con el mapeo de Jordan-Wigner):
  - 4 orbitales espaciales -> 8 spin-orbitales -> 8 qubits.
  - Qubits 0..3 = spin-orbitales alfa, qubits 4..7 = spin-orbitales beta.
  - Orbital moleculares ordenados por energía de Hartree-Fock.
  - El hamiltoniano YA incluye la repulsión nuclear.
"""
import numpy as np
from pyscf import gto, scf, mcscf, ao2mo, cc
from qiskit.quantum_info import SparsePauliOp

N_HIDROGENOS = 4

# Grilla en Å: más densa cerca del mínimo (~0,9-1,0 Å) y en la zona >= 1,8 Å.
# Incluye exactamente 1,8 y 3,0.
DISTANCES = [0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.4, 1.6, 1.8, 2.0, 2.2, 2.4, 2.7, 3.0]


# --------------------------------------------------------------------------
# Jordan-Wigner
# --------------------------------------------------------------------------
def _destruccion(p, n_qubits):
    """Operador de destrucción del spin-orbital p (Jordan-Wigner)."""
    return SparsePauliOp.from_sparse_list(
        [("Z" * p + "X", list(range(p + 1)), 0.5),
         ("Z" * p + "Y", list(range(p + 1)), 0.5j)],
        n_qubits,
    )


def _hamiltoniano_qubits(h, g, constante, norb):
    """H = constante + sum h_pq a†_p a_q + 1/2 sum g_pqrs a†_p a†_r a_s a_q  (notación química)."""
    n_qubits = 2 * norb
    a = [_destruccion(p, n_qubits) for p in range(n_qubits)]
    a_dag = [op.adjoint() for op in a]

    # salto[s][p][q] mueve un electrón de spin s del orbital q al orbital p
    salto = [[[a_dag[s * norb + p].dot(a[s * norb + q]).simplify()
               for q in range(norb)] for p in range(norb)] for s in range(2)]

    terminos = [SparsePauliOp("I" * n_qubits, constante)]
    for s in range(2):
        for p in range(norb):
            for q in range(norb):
                terminos.append(h[p, q] * salto[s][p][q])

    for s in range(2):
        for t in range(2):
            for p in range(norb):
                for q in range(norb):
                    segundo = SparsePauliOp.sum(
                        [g[p, q, r, u] * salto[t][r][u] for r in range(norb) for u in range(norb)])
                    terminos.append(0.5 * salto[s][p][q].dot(segundo))
                    if s == t:
                        # corrección por el orden de operadores (mismo spin)
                        for u in range(norb):
                            terminos.append(-0.5 * g[p, q, q, u] * salto[s][p][u])
    return SparsePauliOp.sum(terminos).simplify(atol=1e-10)


def operador_numero(norb):
    """N = suma de n_p = (I - Z_p)/2 sobre los 2*norb qubits."""
    n = 2 * norb
    terminos = [SparsePauliOp("I" * n, 0.5 * n)]
    for p in range(n):
        terminos.append(SparsePauliOp.from_sparse_list([("Z", [p], -0.5)], n))
    return SparsePauliOp.sum(terminos).simplify()


def operador_s2(norb):
    """S² = S- S+ + Sz (Sz + 1), con alfa en los qubits 0..norb-1 y beta en los siguientes."""
    n = 2 * norb
    a = [_destruccion(p, n) for p in range(n)]
    a_dag = [op.adjoint() for op in a]
    s_mas = SparsePauliOp.sum([a_dag[p].dot(a[norb + p]) for p in range(norb)])
    s_menos = s_mas.adjoint()
    sz = 0.5 * SparsePauliOp.sum(
        [a_dag[p].dot(a[p]) - a_dag[norb + p].dot(a[norb + p]) for p in range(norb)])
    return (s_menos.dot(s_mas) + sz.dot(sz) + sz).simplify(atol=1e-10)


# --------------------------------------------------------------------------
# Problema completo
# --------------------------------------------------------------------------
def geometria(d, n_h=N_HIDROGENOS):
    """Cadena recta de n_h hidrógenos separados por d (Å), sobre el eje z."""
    return "; ".join(f"H 0 0 {i * d:.10f}" for i in range(n_h))


def build_problem(d: float) -> dict:
    """H4 lineal, STO-3G, separación d (Å)."""
    mol = gto.M(atom=geometria(d), basis="sto-3g", verbose=0)
    hf = scf.RHF(mol)
    hf.conv_tol = 1e-12
    hf.run()
    if not hf.converged:
        raise RuntimeError(f"Hartree-Fock no convergió para d={d}")

    norb = mol.nao_nr()
    nelec = mol.nelectron
    n_alfa = n_beta = nelec // 2

    # Integrales en la base de orbitales moleculares (todos los orbitales activos)
    activo = mcscf.CASCI(hf, norb, nelec)
    h1, constante = activo.get_h1cas()          # constante incluye la repulsión nuclear
    h2 = ao2mo.restore(1, activo.get_h2cas(), norb)
    e_fci = float(activo.kernel()[0])

    ccsd = cc.CCSD(hf)
    ccsd.conv_tol = 1e-10
    ccsd.run()
    if not ccsd.converged:
        raise RuntimeError(f"CCSD no convergió para d={d}")

    # Estado de Hartree-Fock: qubits 0..n_alfa-1 (alfa) y norb..norb+n_beta-1 (beta) ocupados.
    # Convención Qiskit: el qubit 0 es el carácter de más a la derecha.
    ocupados = [1 if (q < n_alfa or norb <= q < norb + n_beta) else 0 for q in range(2 * norb)]
    hf_bitstring = "".join(str(b) for b in ocupados[::-1])

    return {
        "d": d,
        "hamiltonian": _hamiltoniano_qubits(h1, h2, constante, norb),
        "n_qubits": 2 * norb,
        "hf_bitstring": hf_bitstring,
        "e_hf": float(hf.e_tot),
        "e_ccsd": float(ccsd.e_tot),
        "e_fci": e_fci,
        "ccsd_t1": np.array(ccsd.t1),   # (n_ocupados, n_virtuales), orbitales espaciales
        "ccsd_t2": np.array(ccsd.t2),   # (o, o, v, v), orbitales espaciales
        "number_op": operador_numero(norb),
        "s2_op": operador_s2(norb),
    }


# --------------------------------------------------------------------------
# Validación
# --------------------------------------------------------------------------
def validate_problem(problem: dict, penalizacion: float = 10.0) -> dict:
    """Autovalor más chico de H restringido a N=4 y S²=0, comparado con el FCI de PySCF.

    Se suma una penalización a (N - 4)² y a S² para aislar el sector de 4 electrones singlete.
    Devuelve el error |E_min - e_fci| y los valores esperados de N y S² en ese estado.
    """
    H = problem["hamiltonian"].to_matrix()
    N = problem["number_op"].to_matrix()
    S2 = problem["s2_op"].to_matrix()
    ne = int(round(sum(int(b) for b in problem["hf_bitstring"])))
    I = np.eye(H.shape[0])
    pen = H + penalizacion * (N - ne * I) @ (N - ne * I) + penalizacion * S2
    w, v = np.linalg.eigh(pen)
    psi = v[:, 0]
    e_min = float(np.real(psi.conj() @ H @ psi))
    return {
        "e_min": e_min,
        "error": abs(e_min - problem["e_fci"]),
        "N": float(np.real(psi.conj() @ N @ psi)),
        "S2": float(np.real(psi.conj() @ S2 @ psi)),
        "e_hf_check": float(np.real(_estado_hf(problem).conj() @ H @ _estado_hf(problem))),
    }


def _estado_hf(problem):
    v = np.zeros(2 ** problem["n_qubits"], dtype=complex)
    v[int(problem["hf_bitstring"], 2)] = 1.0
    return v

In [ ]:
# ---- src/ansatz.py ----
# Dueño: Fede. Solo librerías externas, nunca `from src.x import ...`.


def build_ansatz(problem: dict, kind: str) -> "QuantumCircuit":
    return NotImplemented


def count_cz(circuit: "QuantumCircuit") -> int:
    """Transpila a ['cz','sx','x','rz'] con conectividad en línea y cuenta cz."""
    return NotImplemented

In [ ]:
# ---- src/vqe.py ----
# Dueño: Fede. Solo librerías externas, nunca `from src.x import ...`.


def optimize(problem: dict, ansatz: "QuantumCircuit", theta0=None) -> dict:
    """Devuelve {"theta": ndarray, "e_exact": float, "n_iters": int}."""
    return NotImplemented

In [ ]:
# ---- src/measurement.py ----
# Dueño: Maxi. Solo librerías externas, nunca `from src.x import ...`.
"""Shot-based estimation of expectation values with qiskit_aer's SamplerV2.

Public API (interface in docs/task_split.md):
    measure_energy(circuit, theta, hamiltonian, total_shots) -> {"energy", "sigma", "shots"}

Extras:
    measure_expectation   same thing for any SparsePauliOp (N, S², H², ...)
    validate_sigma        repeat the measurement with different seeds and compare
                          the spread of the energies with the reported sigma
    shots_for_sigma       shots needed to reach a target sigma (sigma ~ 1/sqrt(shots))
    compare_allocations   uniform vs |c_k|-weighted vs adaptive shot allocation
    measure_reliability   <N>, <S²> and Var(H) = <H²> - <H>², measured with shots

Shot allocation strategies (`allocation=`):
    "uniform"   same number of shots for every qubit-wise commuting group
    "weighted"  shots proportional to sum_k |c_k| over the group
    "adaptive"  a pilot run (uniform) estimates the per-shot std sigma_g of every
                group, and the remaining shots go proportional to sigma_g, which
                minimizes sum_g sigma_g² / N_g for a fixed total (Neyman allocation)
"""
import numpy as np
from qiskit import transpile
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import SamplerV2

ALLOCATIONS = ("uniform", "weighted", "adaptive")
MIN_SHOTS_PER_GROUP = 2   # at least 2 shots so that the sample variance is defined
PILOT_FRACTION = 0.1      # fraction of the budget spent on the pilot run in "adaptive"


# --------------------------------------------------------------------------
# Building blocks
# --------------------------------------------------------------------------
def _split_identity(operator):
    """Return (constant, operator without the identity term). The constant needs no shots."""
    operator = operator.simplify(atol=1e-12)
    labels = operator.paulis.to_labels()
    identity = "I" * operator.num_qubits
    constant = float(np.real(sum(c for l, c in zip(labels, operator.coeffs) if l == identity)))
    keep = [i for i, l in enumerate(labels) if l != identity]
    rest = operator[keep] if keep else None
    return constant, rest


def _group_basis(group):
    """Basis per qubit ('I', 'X', 'Y' or 'Z') shared by all terms of a qubit-wise commuting group."""
    n = group.num_qubits
    basis = ["I"] * n
    for label in group.paulis.to_labels():
        for q in range(n):
            p = label[n - 1 - q]  # Qiskit labels: qubit 0 is the rightmost character
            if p != "I":
                basis[q] = p
    return basis


def _measurement_circuit(state_circuit, basis):
    """Append the basis-change rotations and a measurement of every qubit."""
    qc = state_circuit.copy()
    qc.barrier()
    for q, p in enumerate(basis):
        if p == "X":
            qc.h(q)
        elif p == "Y":
            qc.sdg(q)
            qc.h(q)
    qc.measure_all()
    return qc


def _per_shot_values(counts, group):
    """Return (values, weights): value of sum_k c_k P_k for each distinct bitstring, and its count.

    A Pauli string measured in its eigenbasis gives the parity of the bits on its support:
    +1 for an even number of ones, -1 for an odd number.
    """
    coeffs = np.real(group.coeffs)
    support = group.paulis.x | group.paulis.z                 # (n_terms, n_qubits), qubit 0 first
    bitstrings = list(counts)
    # bitstring[-1] is qubit 0, so reverse to get columns in qubit order
    bits = np.array([[int(b) for b in s.replace(" ", "")[::-1]] for s in bitstrings], dtype=int)
    parity = (bits @ support.T.astype(int)) % 2                # (n_outcomes, n_terms)
    values = (1 - 2 * parity) @ coeffs
    weights = np.array([counts[s] for s in bitstrings], dtype=float)
    return values, weights


def _group_stats(counts, group):
    """Mean and per-shot sample variance of the group's observable."""
    values, weights = _per_shot_values(counts, group)
    n = weights.sum()
    mean = np.sum(weights * values) / n
    var = np.sum(weights * (values - mean) ** 2) / max(n - 1, 1)
    return mean, var, n


def _distribute(total, scores):
    """Split `total` shots proportionally to `scores` (largest remainder), at least MIN_SHOTS_PER_GROUP each."""
    k = len(scores)
    scores = np.asarray(scores, dtype=float)
    floor = MIN_SHOTS_PER_GROUP * k
    if total <= floor:
        return np.full(k, max(MIN_SHOTS_PER_GROUP, total // k), dtype=int)
    if scores.sum() <= 0:
        scores = np.ones(k)
    extra = total - floor
    raw = extra * scores / scores.sum()
    shots = np.floor(raw).astype(int)
    remainder = extra - shots.sum()
    shots[np.argsort(raw - shots)[::-1][:remainder]] += 1
    return shots + MIN_SHOTS_PER_GROUP


def _run(sampler, circuits, shots):
    """One SamplerV2 job with a different number of shots per circuit; returns a list of counts."""
    pubs = [(qc, None, int(s)) for qc, s in zip(circuits, shots)]
    result = sampler.run(pubs).result()
    return [r.data.meas.get_counts() for r in result]


# --------------------------------------------------------------------------
# Measurement
# --------------------------------------------------------------------------
def measure_expectation(circuit, theta, operator, total_shots: int,
                        allocation: str = "weighted", seed=None) -> dict:
    """Estimate <operator> with `total_shots` shots spread over qubit-wise commuting groups.

    The sigma accounts for the covariances inside each group exactly, because the
    per-shot value of the whole group (sum_k c_k P_k on one bitstring) is what gets
    averaged. Different groups are independent circuits, so their variances add.

    Returns {"energy", "sigma", "shots", "n_groups", "allocation", "groups"}; "groups"
    has the shots, mean and per-shot std of every group.
    """
    if allocation not in ALLOCATIONS:
        raise ValueError(f"allocation must be one of {ALLOCATIONS}, got {allocation!r}")

    qc = circuit.assign_parameters(theta) if circuit.num_parameters > 0 else circuit
    constant, rest = _split_identity(operator)
    if rest is None:
        return {"energy": constant, "sigma": 0.0, "shots": 0, "n_groups": 0,
                "allocation": allocation, "groups": []}

    groups = rest.group_commuting(qubit_wise=True)
    backend = AerSimulator()
    circuits = transpile([_measurement_circuit(qc, _group_basis(g)) for g in groups], backend)
    # Aer seeds the i-th circuit of a job with seed_simulator + i, so nearby user seeds
    # would reuse random streams across calls. Derive well-separated job seeds instead.
    job_seeds = np.random.SeedSequence(seed).generate_state(2) if seed is not None else None

    def sampler_for(job):
        if job_seeds is None:
            return SamplerV2()
        return SamplerV2(options={"backend_options": {"seed_simulator": int(job_seeds[job])}})

    sampler = sampler_for(0)

    if allocation == "uniform":
        shots = _distribute(total_shots, np.ones(len(groups)))
        counts = _run(sampler, circuits, shots)
    elif allocation == "weighted":
        shots = _distribute(total_shots, [np.sum(np.abs(g.coeffs)) for g in groups])
        counts = _run(sampler, circuits, shots)
    else:  # adaptive
        pilot = _distribute(int(PILOT_FRACTION * total_shots), np.ones(len(groups)))
        pilot_counts = _run(sampler, circuits, pilot)
        stds = np.array([np.sqrt(_group_stats(c, g)[1]) for c, g in zip(pilot_counts, groups)])
        # if a group gave zero spread in the pilot, keep a small weight so it is not starved
        stds = np.maximum(stds, 1e-3 * max(stds.max(), 1e-12))
        # Neyman target for the total budget, minus what each group already got in the pilot
        target = _distribute(total_shots, stds)
        main = np.maximum(target - pilot, 0)
        main_counts = _run(sampler_for(1), circuits, np.maximum(main, 1))
        counts = []
        for pc, mc, m in zip(pilot_counts, main_counts, main):
            merged = dict(pc)
            if m > 0:
                for k, v in mc.items():
                    merged[k] = merged.get(k, 0) + v
            counts.append(merged)
        shots = pilot + main

    energy = constant
    variance = 0.0
    details = []
    for group, c in zip(groups, counts):
        mean, var, n = _group_stats(c, group)
        energy += mean
        variance += var / n
        details.append({"n_terms": len(group), "shots": int(n),
                        "mean": float(mean), "std_per_shot": float(np.sqrt(var))})

    return {
        "energy": float(energy),
        "sigma": float(np.sqrt(variance)),
        "shots": int(sum(d["shots"] for d in details)),
        "n_groups": len(groups),
        "allocation": allocation,
        "groups": details,
    }


def measure_energy(circuit, theta, hamiltonian, total_shots: int,
                   allocation: str = "weighted", seed=None) -> dict:
    """Devuelve {"energy": float, "sigma": float, "shots": int}.

    Groups the Pauli terms into qubit-wise commuting sets, measures one circuit
    per group (shots split according to `allocation`) and returns the energy
    estimate with its statistical standard error. The identity term is added
    exactly and costs no shots.
    """
    r = measure_expectation(circuit, theta, hamiltonian, total_shots, allocation, seed)
    return {"energy": r["energy"], "sigma": r["sigma"], "shots": r["shots"]}


# --------------------------------------------------------------------------
# Validation of sigma and shot budget
# --------------------------------------------------------------------------
def validate_sigma(circuit, theta, hamiltonian, total_shots: int, n_repeats: int = 30,
                   allocation: str = "weighted", seed0: int = 0) -> dict:
    """Repeat measure_energy `n_repeats` times with seeds seed0, seed0+1, ...

    If sigma is right, the standard deviation of the energies ("std_energies")
    matches the reported sigma ("mean_sigma") within the sampling error of a std
    estimated from n samples, about std / sqrt(2 (n - 1)).
    """
    runs = [measure_energy(circuit, theta, hamiltonian, total_shots, allocation, seed=seed0 + i)
            for i in range(n_repeats)]
    energies = np.array([r["energy"] for r in runs])
    sigmas = np.array([r["sigma"] for r in runs])
    std = float(np.std(energies, ddof=1))
    return {
        "allocation": allocation,
        "total_shots": int(runs[0]["shots"]),
        "n_repeats": n_repeats,
        "mean_energy": float(energies.mean()),
        "std_energies": std,
        "std_energies_error": std / np.sqrt(2 * (n_repeats - 1)),
        "mean_sigma": float(sigmas.mean()),
        "ratio": std / float(sigmas.mean()),
        "energies": energies,
        "sigmas": sigmas,
    }


def shots_for_sigma(sigma: float, shots: int, target: float = 1e-3) -> int:
    """Shots needed to reach `target` given that `shots` gave `sigma` (sigma ~ 1/sqrt(shots))."""
    return int(np.ceil(shots * (sigma / target) ** 2))


def compare_allocations(circuit, theta, hamiltonian, total_shots: int, n_repeats: int = 30,
                        target: float = 1e-3, allocations=ALLOCATIONS, seed0: int = 0) -> list:
    """For every allocation: reported sigma, empirical std over repeats, and shots for sigma < target."""
    rows = []
    for allocation in allocations:
        v = validate_sigma(circuit, theta, hamiltonian, total_shots, n_repeats, allocation, seed0)
        rows.append({
            "allocation": allocation,
            "shots": v["total_shots"],
            "mean_energy": v["mean_energy"],
            "mean_sigma": v["mean_sigma"],
            "std_energies": v["std_energies"],
            "ratio": v["ratio"],
            "shots_for_target": shots_for_sigma(v["mean_sigma"], v["total_shots"], target),
        })
    return rows


# --------------------------------------------------------------------------
# Reliability indicators (theory notes, section 11)
# --------------------------------------------------------------------------
def measure_reliability(circuit, theta, hamiltonian, number_op, s2_op, total_shots: int,
                        allocation: str = "weighted", seed=None) -> dict:
    """Measure <N>, <S²> and Var(H) with shots, using the same code as the energy.

    Var(H) = <(H - a)²> - (<H> - a)². Shifting by a = measured <H> keeps the
    coefficients of (H - a)² small, so its estimate is much less noisy than
    measuring <H²> and <H>² separately; the second term is then ~sigma_E² and
    is dropped. Var(H) = 0 if and only if the state is an eigenstate.
    """
    s = None if seed is None else int(seed)
    energy = measure_expectation(circuit, theta, hamiltonian, total_shots, allocation, s)
    a = energy["energy"]
    n_qubits = hamiltonian.num_qubits
    shifted = (hamiltonian - SparsePauliOp("I" * n_qubits, a)).simplify(atol=1e-12)
    h2 = shifted.dot(shifted).simplify(atol=1e-12)
    var = measure_expectation(circuit, theta, h2, total_shots, allocation,
                              None if s is None else s + 10)
    n = measure_expectation(circuit, theta, number_op, total_shots, allocation,
                            None if s is None else s + 20)
    s2 = measure_expectation(circuit, theta, s2_op, total_shots, allocation,
                             None if s is None else s + 30)
    return {
        "energy": a, "energy_sigma": energy["sigma"],
        "variance": var["energy"], "variance_sigma": var["sigma"],
        "N": n["energy"], "N_sigma": n["sigma"],
        "S2": s2["energy"], "S2_sigma": s2["sigma"],
        "shots": energy["shots"] + var["shots"] + n["shots"] + s2["shots"],
    }

In [ ]:
# ---- src/analysis.py ----
# Dueño: Maxi. Lee results/<kind>_d<d>.json y arma tabla, gráficos, R_e y D_e.
# Solo librerías externas, nunca `from src.x import ...`.
"""Results bookkeeping and the final analysis of the potential energy curve.

A *record* is one point of the curve, a JSON-serializable dict:
    {"kind", "d", "energy", "sigma", "shots", "cz", "theta",
     "e_hf", "e_ccsd", "e_fci", ...optional extras (allocation, reliability...)}

Pipeline:
    make_record(...) -> save_record(...)            one point -> results/<kind>_d<d>.json
    load_records(kind)                              all points of one ansatz, sorted by d
    results_table(records)                          pandas table for the notebook
    plot_curve(records), plot_errors(records)       figures
    fit_equilibrium(d, E, sigma)                    R_e and E(R_e) with bootstrap errors
    dissociation_energy(records)                    D_e = E(3.0 Å) - E(R_e)
    beats_ccsd(records)                             |E - E_FCI| + sigma < |E_CCSD - E_FCI| for d >= 1.8
    summary(records)                                all of the above for VQE, CCSD and FCI
"""
import glob
import json
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.interpolate import CubicSpline
from scipy.optimize import minimize_scalar

RESULTS_DIR = "results"
HARTREE_TO_MHA = 1000.0
CHEMICAL_ACCURACY = 1.6e-3   # Ha
D_STRETCHED = 1.8            # Å, from here on we have to beat CCSD
D_DISSOCIATED = 3.0          # Å, atoms considered separated (D_e definition in CONSIGNA.md)


# --------------------------------------------------------------------------
# Records
# --------------------------------------------------------------------------
def make_record(kind: str, problem: dict, theta, measurement: dict, cz: int, **extra) -> dict:
    """Build a record from build_problem's dict, the optimal theta and measure_energy's result."""
    record = {
        "kind": kind,
        "d": float(problem["d"]),
        "energy": float(measurement["energy"]),
        "sigma": float(measurement["sigma"]),
        "shots": int(measurement["shots"]),
        "cz": int(cz),
        "theta": [float(t) for t in np.atleast_1d(theta)] if theta is not None else [],
        "e_hf": float(problem["e_hf"]),
        "e_ccsd": float(problem["e_ccsd"]),
        "e_fci": float(problem["e_fci"]),
    }
    record.update(_to_jsonable(extra))
    return record


def _to_jsonable(obj):
    if isinstance(obj, dict):
        return {k: _to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_to_jsonable(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, np.generic):
        return obj.item()
    return obj


def record_path(kind: str, d: float, results_dir: str = RESULTS_DIR) -> str:
    return os.path.join(results_dir, f"{kind}_d{d:.2f}.json")


def save_record(record: dict, results_dir: str = RESULTS_DIR) -> str:
    os.makedirs(results_dir, exist_ok=True)
    path = record_path(record["kind"], record["d"], results_dir)
    with open(path, "w") as f:
        json.dump(record, f, indent=2)
    return path


def load_records(kind: str, results_dir: str = RESULTS_DIR) -> list:
    records = []
    for path in glob.glob(os.path.join(results_dir, f"{kind}_d*.json")):
        with open(path) as f:
            records.append(json.load(f))
    return sorted(records, key=lambda r: r["d"])


def _arrays(records):
    records = sorted(records, key=lambda r: r["d"])
    get = lambda key: np.array([r[key] for r in records], dtype=float)
    return records, get


# --------------------------------------------------------------------------
# Table
# --------------------------------------------------------------------------
def results_table(records) -> pd.DataFrame:
    """One row per distance: E ± sigma, shots, cz, references and errors in mHa."""
    rows = []
    for r in sorted(records, key=lambda r: r["d"]):
        err = abs(r["energy"] - r["e_fci"])
        err_ccsd = abs(r["e_ccsd"] - r["e_fci"])
        stretched = r["d"] >= D_STRETCHED - 1e-9
        rows.append({
            "d (Å)": r["d"],
            "E (Ha)": r["energy"],
            "σ (mHa)": r["sigma"] * HARTREE_TO_MHA,
            "shots": r["shots"],
            "cz": r["cz"],
            "E_CCSD (Ha)": r["e_ccsd"],
            "E_FCI (Ha)": r["e_fci"],
            "|E-E_FCI| (mHa)": err * HARTREE_TO_MHA,
            "|E_CCSD-E_FCI| (mHa)": err_ccsd * HARTREE_TO_MHA,
            "le gana a CCSD": bool(err + r["sigma"] < err_ccsd) if stretched else None,
            "precisión química": bool(err < CHEMICAL_ACCURACY),
        })
    return pd.DataFrame(rows)


# --------------------------------------------------------------------------
# Plots
# --------------------------------------------------------------------------
def plot_curve(records, ax=None, label="VQE"):
    """E(d) with ±sigma error bars, against HF, CCSD and FCI."""
    records, get = _arrays(records)
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 4.5))
    d = get("d")
    ax.plot(d, get("e_hf"), ":", color="gray", label="HF")
    ax.plot(d, get("e_ccsd"), "s--", color="tab:orange", ms=4, label="CCSD")
    ax.plot(d, get("e_fci"), "-", color="black", lw=1, label="FCI")
    ax.errorbar(d, get("energy"), yerr=get("sigma"), fmt="o", color="tab:blue",
                ms=4, capsize=3, label=f"{label} (±σ)")
    ax.set_xlabel("d (Å)")
    ax.set_ylabel("E (Ha)")
    ax.set_title("H₄ lineal, STO-3G")
    ax.legend()
    ax.grid(alpha=0.3)
    return ax


def plot_errors(records, ax=None, label="VQE"):
    """|E - E_FCI| in mHa (with sigma) and the CCSD error, log scale, chemical accuracy line."""
    records, get = _arrays(records)
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 4.5))
    d = get("d")
    err = np.abs(get("energy") - get("e_fci")) * HARTREE_TO_MHA
    sig = get("sigma") * HARTREE_TO_MHA
    err_ccsd = np.abs(get("e_ccsd") - get("e_fci")) * HARTREE_TO_MHA
    ax.plot(d, err_ccsd, "s--", color="tab:orange", ms=4, label="|E_CCSD − E_FCI|")
    ax.errorbar(d, err, yerr=sig, fmt="o", color="tab:blue", ms=4, capsize=3,
                label=f"|E_{label} − E_FCI| (±σ)")
    ax.plot(d, err + sig, "v", color="tab:blue", ms=4, alpha=0.4, label="|E − E_FCI| + σ")
    ax.axhline(CHEMICAL_ACCURACY * HARTREE_TO_MHA, color="green", ls=":", label="1,6 mHa")
    ax.axvline(D_STRETCHED, color="gray", ls=":", lw=1)
    ax.set_yscale("log")
    ax.set_xlabel("d (Å)")
    ax.set_ylabel("error (mHa)")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3, which="both")
    return ax


# --------------------------------------------------------------------------
# R_e and D_e
# --------------------------------------------------------------------------
def _minimum(d, e, method, n_points):
    """Location and value of the minimum of the curve through (d, e)."""
    if method == "parabola":
        i = int(np.argmin(e))
        half = n_points // 2
        lo = max(0, min(i - half, len(d) - n_points))
        sel = slice(lo, lo + n_points)
        a, b, c = np.polyfit(d[sel], e[sel], 2)
        if a <= 0:
            return d[i], e[i]
        x = -b / (2 * a)
        return x, c - b ** 2 / (4 * a)
    if method == "spline":
        spline = CubicSpline(d, e)
        i = int(np.argmin(e))
        lo, hi = d[max(i - 1, 0)], d[min(i + 1, len(d) - 1)]
        res = minimize_scalar(spline, bounds=(lo, hi), method="bounded")
        return float(res.x), float(res.fun)
    raise ValueError("method must be 'parabola' or 'spline'")


def fit_equilibrium(d, energies, sigmas=None, method: str = "spline", n_points: int = 5,
                    n_bootstrap: int = 2000, seed: int = 0) -> dict:
    """R_e and E(R_e) from a cubic spline through all points (default), or a parabola
    through the n_points closest to the minimum.

    On the 0.1 Å grid the spline lands within ~0.3 mÅ of the dense-grid FCI minimum,
    while the parabola is off by 10 to 30 mÅ because the curve is not symmetric.

    Errors come from a parametric bootstrap: the energies are resampled as
    E_i + sigma_i * N(0, 1) and the fit is repeated. With sigmas=None (CCSD, FCI)
    the errors are zero. This is only the statistical error, the grid spacing
    also limits the precision of R_e.
    """
    d = np.asarray(d, dtype=float)
    e = np.asarray(energies, dtype=float)
    order = np.argsort(d)
    d, e = d[order], e[order]
    r_e, e_min = _minimum(d, e, method, n_points)
    r_e_sigma = e_min_sigma = 0.0
    if sigmas is not None and np.any(np.asarray(sigmas) > 0):
        s = np.asarray(sigmas, dtype=float)[order]
        rng = np.random.default_rng(seed)
        samples = np.array([_minimum(d, e + s * rng.standard_normal(len(e)), method, n_points)
                            for _ in range(n_bootstrap)])
        r_e_sigma, e_min_sigma = samples.std(axis=0, ddof=1)
    return {"R_e": float(r_e), "R_e_sigma": float(r_e_sigma),
            "E_min": float(e_min), "E_min_sigma": float(e_min_sigma), "method": method}


def dissociation_energy(records, key: str = "energy", method: str = "spline",
                        d_dissociated: float = D_DISSOCIATED, **fit_kwargs) -> dict:
    """D_e = E(3.0 Å) - E(R_e) for the curve stored under `key` ("energy", "e_ccsd" or "e_fci")."""
    records, get = _arrays(records)
    d = get("d")
    e = get(key)
    s = get("sigma") if key == "energy" else None
    j = int(np.argmin(np.abs(d - d_dissociated)))
    if abs(d[j] - d_dissociated) > 1e-6:
        raise ValueError(f"no point at d = {d_dissociated} Å")
    fit = fit_equilibrium(d, e, s, method=method, **fit_kwargs)
    s_far = 0.0 if s is None else s[j]
    return {
        **fit,
        "E_dissociated": float(e[j]),
        "D_e": float(e[j] - fit["E_min"]),
        "D_e_sigma": float(np.hypot(s_far, fit["E_min_sigma"])),
    }


def beats_ccsd(records, d_min: float = D_STRETCHED) -> dict:
    """Check |E - E_FCI| + sigma < |E_CCSD - E_FCI| on every point with d >= d_min."""
    rows = []
    for r in sorted(records, key=lambda r: r["d"]):
        if r["d"] < d_min - 1e-9:
            continue
        lhs = abs(r["energy"] - r["e_fci"]) + r["sigma"]
        rhs = abs(r["e_ccsd"] - r["e_fci"])
        rows.append({"d": r["d"], "lhs_mHa": lhs * HARTREE_TO_MHA, "rhs_mHa": rhs * HARTREE_TO_MHA,
                     "ok": bool(lhs < rhs),
                     "chemical_accuracy": bool(abs(r["energy"] - r["e_fci"]) < CHEMICAL_ACCURACY)})
    return {"all_ok": bool(rows) and all(r["ok"] for r in rows),
            "all_chemical_accuracy": bool(rows) and all(r["chemical_accuracy"] for r in rows),
            "points": rows}


def summary(records, method: str = "spline", **fit_kwargs) -> pd.DataFrame:
    """R_e and D_e for VQE (with errors), CCSD and FCI, side by side."""
    rows = []
    for name, key in (("VQE", "energy"), ("CCSD", "e_ccsd"), ("FCI", "e_fci")):
        r = dissociation_energy(records, key=key, method=method, **fit_kwargs)
        rows.append({"método": name,
                     "R_e (Å)": r["R_e"], "σ R_e (Å)": r["R_e_sigma"],
                     "E(R_e) (Ha)": r["E_min"],
                     "D_e (Ha)": r["D_e"], "σ D_e (mHa)": r["D_e_sigma"] * HARTREE_TO_MHA})
    return pd.DataFrame(rows)


def report(records, method: str = "spline"):
    """Everything the notebook shows at the end: table, figures, R_e/D_e and the CCSD check."""
    table = results_table(records)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
    plot_curve(records, axes[0])
    plot_errors(records, axes[1])
    fig.tight_layout()
    plt.close(fig)   # do not auto-display here; show it with out["figure"]
    check = beats_ccsd(records)
    return {"table": table, "summary": summary(records, method=method),
            "beats_ccsd": check, "figure": fig}

In [ ]:
print(f"Tiempo total de ejecución: {(time.time() - _notebook_t0) / 60:.1f} min")